In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
from datasets import Dataset

# ── Load your Excel CSV ───────────────────────────────────────────
df = pd.read_csv("/content/data.csv")

# ── Convert each row to text ──────────────────────────────────────
def row_to_text(row):
    return f"""Customer {row['CustomerName']} from {row['City']} ordered {row['Quantity']} units of {row['ProductName']}. Total amount was {row['TotalAmount']} USD. Payment method was {row['PaymentMethod']} and the order was {row['OrderStatus']}."""

df["text"] = df.apply(row_to_text, axis=1)

# ── Keep only text column ─────────────────────────────────────────
df = df[["text"]]

# ── Convert to HuggingFace Dataset ────────────────────────────────
dataset = Dataset.from_pandas(df)

print(f"✅ Dataset ready: {len(dataset)} records")
print(dataset[0])

✅ Dataset ready: 100000 records
{'text': 'Customer Vihaan Sharma from Washington ordered 3 units of Drone Mini. Total amount was 319.86 USD. Payment method was Debit Card and the order was Delivered.'}


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")

# Add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

dataset = dataset.map(tokenize)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto"
)

print(f"✅ Model ready")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model ready


In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Prepare model for training ────────────────────────────────────
model = prepare_model_for_kbit_training(model)

# ── LoRA Config ───────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# ── Apply LoRA to model ───────────────────────────────────────────
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
# trainable params: 2,097,152 || all params: 1,543,714,816 || trainable%: 0.14%

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [6]:
# ── Split train/test ──────────────────────────────────────────────
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"✅ Train: {len(train_dataset)} records")
print(f"✅ Eval:  {len(eval_dataset)} records")

# ── Tokenize ──────────────────────────────────────────────────────
train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset  = eval_dataset.map(tokenize, batched=True)

print(f"✅ Dataset ready!")

✅ Train: 90000 records
✅ Eval:  10000 records


Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

✅ Dataset ready!


In [12]:
from trl import SFTTrainer, SFTConfig

# ── Training Config ───────────────────────────────────────────────
training_args = SFTConfig(
    output_dir="./lora-output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    save_steps=500,
    max_length=512,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
    dataset_text_field="text",
)

# ── Trainer ───────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,       # ✅ moved here
)

print("🚀 Starting LoRA training...")
trainer.train()
print("✅ Training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Starting LoRA training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ── Save LoRA weights ─────────────────────────────────────────────
model.save_pretrained("./lora-finetuned")
tokenizer.save_pretrained("./lora-finetuned")

print("✅ Model saved to ./lora-finetuned")